In [ ]:
!pip -q install geopandas shapely pyproj scipy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from __future__ import annotations
from pathlib import Path
import math, random, re, json, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
from scipy.ndimage import gaussian_filter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
DEVICE


device(type='cuda')

In [ ]:
INPUT_BASE = Path('/content/drive/MyDrive/Research')
BASE = Path('/content/drive/MyDrive/version_2')
DRIVE_ROOT = Path('/content/drive/MyDrive')


def resolve_file(*patterns):
    search_roots = [INPUT_BASE, BASE, DRIVE_ROOT]
    seen = set()
    for root in search_roots:
        if root in seen or not root.exists():
            continue
        seen.add(root)
        for pattern in patterns:
            matches = sorted(root.rglob(pattern))
            if matches:
                print(f'Resolved {pattern}: {matches[0]}')
                return matches[0]
    raise FileNotFoundError(patterns)

def stage_local_file(src_path, cache_dir=Path('/content/ds340w_cache')):
    cache_dir.mkdir(parents=True, exist_ok=True)
    dst_path = cache_dir / Path(src_path).name
    try:
        same_size = dst_path.exists() and dst_path.stat().st_size == Path(src_path).stat().st_size
    except OSError:
        same_size = False
    if same_size:
        print(f'Using cached local file: {dst_path}')
        return dst_path
    print(f'Copying to local runtime storage: {src_path} -> {dst_path}')
    shutil.copy2(src_path, dst_path)
    print(f'Local copy ready: {dst_path}')
    return dst_path

DATA_PATH = resolve_file('Updated_Crime_Enriched_PointLevel_v16c.csv', '*v16c*.csv')
LOCAL_DATA_PATH = stage_local_file(DATA_PATH)
NYC_SHP = resolve_file('nybb.shp', 'nycbdry/nybb.shp', 'Combined/nycbdry/nybb.shp')
OUT_DIR = BASE / 'pointlevel_outputs_v16c_version_2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
HEATMAP_DIR = OUT_DIR / 'predicted_hourly_heatmaps'
ACTUAL_HEATMAP_DIR = OUT_DIR / 'actual_hourly_heatmaps'
for _p in [HEATMAP_DIR, ACTUAL_HEATMAP_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

TARGET_CRIME = None
STUDY_START = '2022-01-01'
GRID_M = 2000.0
PATROL_TILE_M = 2000.0
LOOKBACK_HOURS = 72
VAL_DAYS = 3
TEST_DAYS = 3
TEST_HOURS = 24 * TEST_DAYS
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
TEMPORAL_NODE_CHUNK = 2048
EPOCHS = 12
LEARNING_RATE = 1e-3
PATIENCE = 2
K_NEIGHBORS = 8
KDE_SIGMA = 0.85
RECENT_HOUR_BLEND = 0.10
RECENT_DAY_BLEND = 0.20

print('DATA_PATH =', DATA_PATH)
print('INPUT_BASE =', INPUT_BASE)
print('LOCAL_DATA_PATH =', LOCAL_DATA_PATH)
print('NYC_SHP =', NYC_SHP)
print('OUT_DIR =', OUT_DIR)


Resolved Updated_Crime_Enriched_PointLevel_v16c.csv: /content/drive/MyDrive/Research/Updated_Crime_Enriched_PointLevel_v16c.csv
Copying to local runtime storage: /content/drive/MyDrive/Research/Updated_Crime_Enriched_PointLevel_v16c.csv -> /content/ds340w_cache/Updated_Crime_Enriched_PointLevel_v16c.csv
Local copy ready: /content/ds340w_cache/Updated_Crime_Enriched_PointLevel_v16c.csv
Resolved nybb.shp: /content/drive/MyDrive/Research/nycbdry/nybb.shp
DATA_PATH = /content/drive/MyDrive/Research/Updated_Crime_Enriched_PointLevel_v16c.csv
INPUT_BASE = /content/drive/MyDrive/Research
LOCAL_DATA_PATH = /content/ds340w_cache/Updated_Crime_Enriched_PointLevel_v16c.csv
NYC_SHP = /content/drive/MyDrive/Research/nycbdry/nybb.shp
OUT_DIR = /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2


In [ ]:
def parse_datetime(df):
    if 'occurred_dt' in df.columns:
        dt = pd.to_datetime(df['occurred_dt'], errors='coerce', utc=True)
        if dt.notna().any():
            return dt.dt.tz_convert(None)
    if 'crime_date' in df.columns and 'crime_time' in df.columns:
        dt = pd.to_datetime(df['crime_date'].astype(str) + ' ' + df['crime_time'].astype(str), errors='coerce', utc=True)
        if dt.notna().any():
            return dt.dt.tz_convert(None)
    if 'cmplnt_fr_dt' in df.columns and 'cmplnt_fr_tm' in df.columns:
        dt = pd.to_datetime(df['cmplnt_fr_dt'].astype(str) + ' ' + df['cmplnt_fr_tm'].astype(str), errors='coerce', utc=True)
        return dt.dt.tz_convert(None)
    raise ValueError('No usable datetime columns found.')

def clean_precinct(x):
    if pd.isna(x):
        return None
    s = re.sub(r'[^0-9]', '', str(x))
    return s if s else None

def build_grid(bounds, geom, grid_m):
    minx, miny, maxx, maxy = bounds
    xs = np.arange(minx + grid_m / 2, maxx, grid_m)
    ys = np.arange(miny + grid_m / 2, maxy, grid_m)
    xx, yy = np.meshgrid(xs, ys)
    flat_x = xx.ravel(); flat_y = yy.ravel()
    mask = np.array([geom.contains(Point(x, y)) for x, y in zip(flat_x, flat_y)])
    flat_x = flat_x[mask]; flat_y = flat_y[mask]
    col = ((flat_x - (minx + grid_m / 2)) / grid_m).round().astype(int)
    row = ((flat_y - (miny + grid_m / 2)) / grid_m).round().astype(int)
    return {'x': flat_x, 'y': flat_y, 'row': row, 'col': col, 'cell_id': np.arange(len(flat_x), dtype=np.int64), 'minx': minx, 'miny': miny, 'nrows': len(ys), 'ncols': len(xs), 'grid_m': grid_m}

def assign_cells(x, y, grid):
    col = ((x - (grid['minx'] + grid['grid_m'] / 2)) / grid['grid_m']).round().astype(int)
    row = ((y - (grid['miny'] + grid['grid_m'] / 2)) / grid['grid_m']).round().astype(int)
    key = row.astype(np.int64) * (grid['ncols'] + 1) + col.astype(np.int64)
    gkey = grid['row'].astype(np.int64) * (grid['ncols'] + 1) + grid['col'].astype(np.int64)
    lookup = dict(zip(gkey, grid['cell_id']))
    return np.array([lookup.get(k, -1) for k in key], dtype=np.int64)

def build_knn_adj(coords, k=8):
    d2 = np.sum((coords[:, None, :] - coords[None, :, :]) ** 2, axis=-1)
    idx = np.argsort(d2, axis=1)[:, 1:k+1]
    n = coords.shape[0]
    adj = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        adj[i, idx[i]] = 1.0
    adj = np.maximum(adj, adj.T); np.fill_diagonal(adj, 1.0)
    deg = adj.sum(axis=1)
    deg_inv_sqrt = np.power(deg, -0.5)
    deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0
    D = np.diag(deg_inv_sqrt)
    return D @ adj @ D

def build_hour_features(ts):
    hour = ts.hour; dow = ts.dayofweek; month = ts.month
    return pd.DataFrame({'hour_sin': np.sin(2 * math.pi * hour / 24), 'hour_cos': np.cos(2 * math.pi * hour / 24), 'dow_sin': np.sin(2 * math.pi * dow / 7), 'dow_cos': np.cos(2 * math.pi * dow / 7), 'month_sin': np.sin(2 * math.pi * (month - 1) / 12), 'month_cos': np.cos(2 * math.pi * (month - 1) / 12), 'weekend': (dow >= 5).astype(int)}, index=ts)

def sector_id_from_xy(x, y, minx, miny, tile_m):
    tx = ((x - minx) / tile_m).astype(int)
    ty = ((y - miny) / tile_m).astype(int)
    return ty * 100000 + tx

def render_heatmap(arr, mask, title, out_path, vmax=None):
    sm = gaussian_filter(np.nan_to_num(arr, nan=0.0), sigma=KDE_SIGMA)
    sm[~mask] = np.nan
    plt.figure(figsize=(7, 7))
    plt.imshow(sm, origin='lower', cmap='hot', vmin=0.0, vmax=vmax)
    plt.title(title); plt.axis('off'); plt.colorbar(fraction=0.04, pad=0.02)
    plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.close()

def format_hour_name(ts):
    return ts.strftime('%Y%m%d_%H00')


In [ ]:
usecols = [
    'row_id', 'crime_date', 'crime_time', 'occurred_dt', 'cmplnt_fr_dt', 'cmplnt_fr_tm',
    'ofns_desc', 'crime_precinct', 'addr_pct_cd', 'latitude', 'longitude',
    'holiday', 'weekend', 'is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday',
    't_mean', 'rh_mean', 'wind_mean', 'has_event', 'event_match_count', 'has_road_closure', 'road_closure_count'
]
raw = pd.read_csv(LOCAL_DATA_PATH, usecols=lambda c: c in usecols, low_memory=False)
if 'row_id' in raw.columns:
    raw = raw.loc[pd.to_numeric(raw['row_id'], errors='coerce').fillna(-1) != 0].copy()
raw['dt'] = parse_datetime(raw)
raw = raw.dropna(subset=['dt', 'latitude', 'longitude'])
raw['dt'] = raw['dt'].dt.floor('H')
raw['precinct'] = raw['crime_precinct'].fillna(raw['addr_pct_cd']).map(clean_precinct)
if TARGET_CRIME is not None:
    raw = raw[raw['ofns_desc'].astype(str).str.upper().str.contains(TARGET_CRIME.upper(), na=False)].copy()
study_start_dt = pd.Timestamp(STUDY_START)
raw = raw[raw['dt'] >= study_start_dt].copy()
data_start = raw['dt'].min().floor('H')
end_hour = raw['dt'].max().floor('H')
test_start = end_hour - pd.Timedelta(hours=TEST_HOURS - 1)
val_start = test_start - pd.Timedelta(days=VAL_DAYS)
train_start = data_start
warmup_start = data_start
raw = raw[(raw['dt'] >= warmup_start) & (raw['dt'] <= end_hour)].copy()
print('Filtered rows:', raw.shape)
print('Study start target:', study_start_dt)
print('Data start:', data_start)
print('Train start:', train_start)
print('Val start:', val_start)
print('Test start:', test_start)
print('End hour:', end_hour)


/tmp/ipykernel_836/1789653341.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  raw['dt'] = raw['dt'].dt.floor('H')


Filtered rows: (1648292, 26)
Study start target: 2022-01-01 00:00:00
Data start: 2022-01-01 00:00:00
Train start: 2022-01-01 00:00:00
Val start: 2024-12-26 00:00:00
Test start: 2024-12-29 00:00:00
End hour: 2024-12-31 23:00:00


/tmp/ipykernel_836/1789653341.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  data_start = raw['dt'].min().floor('H')
/tmp/ipykernel_836/1789653341.py:19: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  end_hour = raw['dt'].max().floor('H')


In [ ]:
boroughs = gpd.read_file(NYC_SHP).to_crs(epsg=2263)
nyc_geom = unary_union(boroughs.geometry)
grid = build_grid(nyc_geom.bounds, nyc_geom, GRID_M)
gdf = gpd.GeoDataFrame(raw, geometry=gpd.points_from_xy(raw['longitude'], raw['latitude']), crs='EPSG:4326').to_crs(epsg=2263)
gdf = gdf[gdf.geometry.within(nyc_geom)].copy()
gdf['cell_id'] = assign_cells(gdf.geometry.x.values, gdf.geometry.y.values, grid)
gdf = gdf[gdf['cell_id'] >= 0].copy()
cells = pd.DataFrame({'cell_id': grid['cell_id'], 'x': grid['x'], 'y': grid['y']})
cells['x_norm'] = (cells['x'] - cells['x'].min()) / (cells['x'].max() - cells['x'].min() + 1e-9)
cells['y_norm'] = (cells['y'] - cells['y'].min()) / (cells['y'].max() - cells['y'].min() + 1e-9)
cell_points = gpd.GeoDataFrame(cells[['cell_id']], geometry=gpd.points_from_xy(cells['x'], cells['y']), crs='EPSG:2263')
cell_boro = gpd.sjoin(cell_points, boroughs[['BoroName', 'geometry']], predicate='within', how='left')
cells['borough'] = cell_boro['BoroName'].fillna('Unknown').values
boro_dummies = pd.get_dummies(cells['borough'], prefix='boro')
cells = pd.concat([cells, boro_dummies], axis=1)
static_cols = ['x_norm', 'y_norm'] + list(boro_dummies.columns)
for col in ['t_mean', 'rh_mean', 'wind_mean', 'holiday', 'weekend', 'is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday', 'has_event', 'event_match_count', 'has_road_closure', 'road_closure_count']:
    if col not in gdf.columns:
        gdf[col] = 0
hour_index = pd.date_range(warmup_start, end_hour, freq='h')
hour_feat = build_hour_features(hour_index).reset_index().rename(columns={'index': 'dt'})
hourly_global = gdf.groupby('dt', as_index=False).agg(
    t_mean=('t_mean', 'mean'), rh_mean=('rh_mean', 'mean'), wind_mean=('wind_mean', 'mean'),
    holiday=('holiday', 'max'),
    is_christian_holiday=('is_christian_holiday', 'max'),
    is_islamic_holiday=('is_islamic_holiday', 'max'),
    is_jewish_holiday=('is_jewish_holiday', 'max'),
    is_hindu_holiday=('is_hindu_holiday', 'max')
)
hourly_global = hour_feat.merge(hourly_global, on='dt', how='left').fillna(0)
hourly_cell = gdf.groupby(['dt', 'cell_id'], as_index=False).agg(
    crime_count=('ofns_desc', 'size'), has_event=('has_event', 'max'), event_match_count=('event_match_count', 'max'),
    has_road_closure=('has_road_closure', 'max'), road_closure_count=('road_closure_count', 'max')
)
T = len(hour_index)
M = len(cells)
hour_to_idx = {t: i for i, t in enumerate(hour_index)}
cell_to_idx = {cid: i for i, cid in enumerate(cells['cell_id'].values)}
dynamic_cols = ['crime_count', 'has_event', 'event_match_count', 'has_road_closure', 'road_closure_count']
global_cols = ['t_mean', 'rh_mean', 'wind_mean', 'holiday', 'weekend', 'is_christian_holiday', 'is_islamic_holiday', 'is_jewish_holiday', 'is_hindu_holiday', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
feature_cols = ['crime_count'] + static_cols + [c for c in dynamic_cols if c != 'crime_count'] + global_cols
X_raw = np.zeros((T, M, len(feature_cols)), dtype=np.float32)
y_counts = np.zeros((T, M), dtype=np.float32)
static_vals = cells[static_cols].to_numpy(dtype=np.float32)
for fi in range(len(static_cols)):
    X_raw[:, :, fi + 1] = static_vals[:, fi]
hourly_global = hourly_global.set_index('dt').reindex(hour_index).fillna(0)
for col in global_cols:
    fi = feature_cols.index(col)
    X_raw[:, :, fi] = hourly_global[col].to_numpy(dtype=np.float32)[:, None]
for row in hourly_cell.itertuples(index=False):
    ti = hour_to_idx.get(row.dt)
    ci = cell_to_idx.get(row.cell_id)
    if ti is None or ci is None:
        continue
    y_counts[ti, ci] = float(row.crime_count)
    X_raw[ti, ci, 0] = np.log1p(float(row.crime_count))
    X_raw[ti, ci, feature_cols.index('has_event')] = float(row.has_event)
    X_raw[ti, ci, feature_cols.index('event_match_count')] = np.log1p(float(row.event_match_count))
    X_raw[ti, ci, feature_cols.index('has_road_closure')] = float(row.has_road_closure)
    X_raw[ti, ci, feature_cols.index('road_closure_count')] = np.log1p(float(row.road_closure_count))
mask = np.zeros((grid['nrows'], grid['ncols']), dtype=bool)
mask[grid['row'], grid['col']] = True
print('Hour count:', T, 'Cell count:', M, 'Feature count:', len(feature_cols))


Hour count: 26304 Cell count: 2102 Feature count: 27


In [ ]:
train_end = val_start - pd.Timedelta(hours=1)
val_end = test_start - pd.Timedelta(hours=1)
train_target_idx = [i for i, t in enumerate(hour_index) if data_start <= t <= train_end]
val_target_idx = [i for i, t in enumerate(hour_index) if val_start <= t <= val_end]
test_target_idx = [i for i, t in enumerate(hour_index) if test_start <= t <= end_hour]
train_sample_idx = [i for i in train_target_idx if i >= LOOKBACK_HOURS]
val_sample_idx = [i for i in val_target_idx if i >= LOOKBACK_HOURS]
test_sample_idx = [i for i in test_target_idx if i >= LOOKBACK_HOURS]
continuous_cols = ['crime_count', 'event_match_count', 'road_closure_count', 't_mean', 'rh_mean', 'wind_mean', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
cont_idx = [feature_cols.index(c) for c in continuous_cols if c in feature_cols]
sum_vec = np.zeros(len(cont_idx), dtype=np.float64)
sumsq_vec = np.zeros(len(cont_idx), dtype=np.float64)
n_obs = 0
stats_chunk = max(24, 24 * 7)
for start in range(0, len(train_target_idx), stats_chunk):
    idx_chunk = train_target_idx[start:start + stats_chunk]
    block = X_raw[idx_chunk][:, :, cont_idx].astype(np.float64, copy=False)
    flat = block.reshape(-1, len(cont_idx))
    sum_vec += flat.sum(axis=0)
    sumsq_vec += np.square(flat).sum(axis=0)
    n_obs += flat.shape[0]
mean_vec = sum_vec / max(n_obs, 1)
var_vec = np.maximum(sumsq_vec / max(n_obs, 1) - np.square(mean_vec), 0.0)
std_vec = np.sqrt(var_vec) + 1e-6
norm_chunk = max(24, 24 * 7)
for start in range(0, T, norm_chunk):
    end = min(T, start + norm_chunk)
    X_raw[start:end, :, cont_idx] = (X_raw[start:end, :, cont_idx] - mean_vec.astype(np.float32)) / std_vec.astype(np.float32)
X = X_raw
coords = cells[['x_norm', 'y_norm']].to_numpy(dtype=np.float32)
adj_norm = build_knn_adj(coords, k=K_NEIGHBORS)
adj_t = torch.tensor(adj_norm, dtype=torch.float32, device=DEVICE)
print('Train samples:', len(train_sample_idx), 'Val samples:', len(val_sample_idx), 'Test samples:', len(test_sample_idx))


Train samples: 26088 Val samples: 72 Test samples: 72


In [ ]:
class ProbSparseAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4, dropout=0.1, factor=5):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.factor = factor
        self.scale = self.head_dim ** -0.5
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        b, t, _ = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        sparsity = scores.max(dim=-1).values - scores.mean(dim=-1)
        u = min(t, max(1, int(self.factor * math.log(t + 1))))
        top_idx = torch.topk(sparsity, k=u, dim=-1).indices
        v_mean = v.mean(dim=-2, keepdim=True)
        context = v_mean.repeat(1, 1, t, 1)
        top_scores = scores.gather(-2, top_idx.unsqueeze(-1).repeat(1, 1, 1, t))
        top_attn = torch.softmax(top_scores, dim=-1)
        top_ctx = torch.matmul(top_attn, v)
        context.scatter_(-2, top_idx.unsqueeze(-1).repeat(1, 1, 1, self.head_dim), top_ctx)
        context = context.transpose(1, 2).contiguous().view(b, t, self.d_model)
        return self.out_proj(self.drop(context))

class InformerEncoderLayer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn = ProbSparseAttention(d_model=d_model, n_heads=n_heads, dropout=dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        x = self.norm2(x + self.ff(x))
        return x

class InformerEncoder(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([InformerEncoderLayer(d_model=d_model, n_heads=n_heads, dropout=dropout) for _ in range(n_layers)])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

class GraphConv(nn.Module):
    def __init__(self, in_feats, out_feats):
        super().__init__()
        self.lin = nn.Linear(in_feats, out_feats)
    def forward(self, x, adj):
        h = torch.matmul(adj, x)
        return self.lin(h)

class STGCNBlock(nn.Module):
    def __init__(self, in_feats, hidden_feats, dropout=0.1):
        super().__init__()
        self.gcn = GraphConv(in_feats, hidden_feats)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_feats)
    def forward(self, x, adj):
        h = self.gcn(x, adj)
        h = self.act(h)
        h = self.drop(h)
        return self.norm(h)

class InformerSTGCNPointModel(nn.Module):
    def __init__(self, feat_dim, d_model=64, n_heads=4, dropout=0.1, temporal_node_chunk=2048):
        super().__init__()
        self.temporal_in = nn.Linear(feat_dim, d_model)
        self.informer = InformerEncoder(d_model=d_model, n_heads=n_heads, n_layers=2, dropout=dropout)
        self.stgcn = STGCNBlock(in_feats=d_model, hidden_feats=d_model, dropout=dropout)
        self.decoder = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.ReLU(), nn.Linear(d_model, 1))
        self.temporal_node_chunk = temporal_node_chunk
    def forward(self, x, adj):
        B, L, M, feat_dim = x.shape
        h = self.temporal_in(x)
        h = h.permute(0, 2, 1, 3).reshape(B * M, L, -1)
        if h.shape[0] > self.temporal_node_chunk:
            parts = []
            for start in range(0, h.shape[0], self.temporal_node_chunk):
                parts.append(self.informer(h[start:start + self.temporal_node_chunk])[:, -1, :])
            temporal_h = torch.cat(parts, dim=0)
        else:
            temporal_h = self.informer(h)[:, -1, :]
        temporal_h = temporal_h.reshape(B, M, -1)
        spatial_h = self.stgcn(temporal_h, adj)
        fused = torch.cat([temporal_h, spatial_h], dim=-1)
        out = self.decoder(fused).squeeze(-1)
        return F.softplus(out)

def poisson_loss(pred, target):
    return torch.mean(pred - target * torch.log(pred + 1e-6))

def evaluate_np(pred, truth):
    mae = np.mean(np.abs(pred - truth))
    rmse = np.sqrt(np.mean((pred - truth) ** 2))
    return mae, rmse


In [ ]:
model = InformerSTGCNPointModel(feat_dim=len(feature_cols), d_model=64, n_heads=4, dropout=0.1, temporal_node_chunk=TEMPORAL_NODE_CHUNK).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
USE_AUTOCAST = DEVICE.type == 'cuda'
AUTOCAST_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32
def build_batch(sample_indices):
    xb = np.stack([X[idx - LOOKBACK_HOURS:idx] for idx in sample_indices], axis=0).astype(np.float32, copy=False)
    yb = y_counts[sample_indices].astype(np.float32, copy=False)
    return xb, yb
def run_batches(sample_indices, train=True, collect_outputs=False):
    order = np.array(sample_indices, dtype=np.int64)
    if train:
        np.random.shuffle(order)
    losses = []
    preds = []
    trues = []
    if train:
        opt.zero_grad(set_to_none=True)
    for step_idx, start in enumerate(range(0, len(order), BATCH_SIZE), start=1):
        batch_idx = order[start:start+BATCH_SIZE]
        xb_np, yb_np = build_batch(batch_idx)
        xb = torch.tensor(xb_np, dtype=torch.float32, device=DEVICE)
        yb = torch.tensor(yb_np, dtype=torch.float32, device=DEVICE)
        if train:
            model.train()
            with torch.autocast(device_type='cuda', dtype=AUTOCAST_DTYPE, enabled=USE_AUTOCAST):
                pred = model(xb, adj_t)
                loss = poisson_loss(pred, yb)
            loss_value = loss.item()
            (loss / GRAD_ACCUM_STEPS).backward()
            if (step_idx % GRAD_ACCUM_STEPS == 0) or (start + BATCH_SIZE >= len(order)):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                opt.zero_grad(set_to_none=True)
        else:
            model.eval()
            with torch.no_grad():
                with torch.autocast(device_type='cuda', dtype=AUTOCAST_DTYPE, enabled=USE_AUTOCAST):
                    pred = model(xb, adj_t)
                    loss = poisson_loss(pred, yb)
            loss_value = loss.item()
        losses.append(loss_value)
        if step_idx % 100 == 0:
            print(("train" if train else "val"), "step", step_idx, "of", math.ceil(len(order) / BATCH_SIZE))
        if collect_outputs:
            preds.append(pred.detach().cpu().numpy())
            trues.append(yb.detach().cpu().numpy())
        del xb, yb, xb_np, yb_np, pred, loss
    pred_out = np.concatenate(preds, axis=0) if collect_outputs and preds else None
    true_out = np.concatenate(trues, axis=0) if collect_outputs and trues else None
    return float(np.mean(losses)), pred_out, true_out

best_val = float('inf'); best_state = None; patience_left = PATIENCE
for epoch in range(1, EPOCHS + 1):
    train_loss, _, _ = run_batches(train_sample_idx, train=True, collect_outputs=False)
    val_loss, val_pred, val_true = run_batches(val_sample_idx, train=False, collect_outputs=True)
    val_mae, val_rmse = evaluate_np(val_pred, val_true)
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_mae={val_mae:.4f} | val_rmse={val_rmse:.4f}')
    if val_loss < best_val:
        best_val = val_loss; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; patience_left = PATIENCE
    else:
        patience_left -= 1
        if patience_left <= 0:
            print('Early stopping triggered.'); break
if best_state is not None:
    model.load_state_dict(best_state)


train step 100 of 6522
train step 200 of 6522
train step 300 of 6522
train step 400 of 6522
train step 500 of 6522
train step 600 of 6522
train step 700 of 6522
train step 800 of 6522
train step 900 of 6522
train step 1000 of 6522
train step 1100 of 6522
train step 1200 of 6522
train step 1300 of 6522
train step 1400 of 6522
train step 1500 of 6522
train step 1600 of 6522
train step 1700 of 6522
train step 1800 of 6522
train step 1900 of 6522
train step 2000 of 6522
train step 2100 of 6522
train step 2200 of 6522
train step 2300 of 6522
train step 2400 of 6522
train step 2500 of 6522
train step 2600 of 6522
train step 2700 of 6522
train step 2800 of 6522
train step 2900 of 6522
train step 3000 of 6522
train step 3100 of 6522
train step 3200 of 6522
train step 3300 of 6522
train step 3400 of 6522
train step 3500 of 6522
train step 3600 of 6522
train step 3700 of 6522
train step 3800 of 6522
train step 3900 of 6522
train step 4000 of 6522
train step 4100 of 6522
train step 4200 of 6522
t

In [ ]:
model.eval()
count_mean = mean_vec[continuous_cols.index('crime_count')]
count_std = std_vec[continuous_cols.index('crime_count')]
def denorm_count_from_feature(col):
    return np.expm1(np.clip(col * count_std + count_mean, a_min=-20.0, a_max=20.0))
start_test_idx = hour_to_idx[test_start]
rolling = X[start_test_idx - LOOKBACK_HOURS:start_test_idx].copy()
test_pred = []
for h in range(TEST_HOURS):
    xb = torch.tensor(rolling[None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        model_pred = model(xb, adj_t).cpu().numpy()[0]
    last_hour = denorm_count_from_feature(rolling[-1, :, 0])
    day_lag = denorm_count_from_feature(rolling[-24, :, 0]) if rolling.shape[0] >= 24 else last_hour
    blended_pred = ((1.0 - RECENT_HOUR_BLEND - RECENT_DAY_BLEND) * model_pred) + (RECENT_HOUR_BLEND * last_hour) + (RECENT_DAY_BLEND * day_lag)
    pred = np.clip(blended_pred, a_min=0.0, a_max=None)
    test_pred.append(pred)
    next_idx = start_test_idx + h
    next_row = X[next_idx].copy()
    next_row[:, 0] = (np.log1p(pred) - count_mean) / count_std
    rolling = np.concatenate([rolling[1:], next_row[None, ...]], axis=0)
test_pred = np.stack(test_pred)
test_true = y_counts[start_test_idx:start_test_idx + TEST_HOURS]
test_mae, test_rmse = evaluate_np(test_pred, test_true)
summary_df = pd.DataFrame([{'target_crime': TARGET_CRIME or 'ALL', 'grid_m': GRID_M, 'patrol_tile_m': PATROL_TILE_M, 'lookback_hours': LOOKBACK_HOURS, 'test_hours': TEST_HOURS, 'kde_sigma': KDE_SIGMA, 'recent_hour_blend': RECENT_HOUR_BLEND, 'recent_day_blend': RECENT_DAY_BLEND, 'test_mae': test_mae, 'test_rmse': test_rmse}])
summary_df.to_csv(OUT_DIR / 'heatmap_model_summary.csv', index=False)
summary_df


,target_crime,grid_m,patrol_tile_m,lookback_hours,test_hours,kde_sigma,recent_hour_blend,recent_day_blend,test_mae,test_rmse
0,ALL,2000.0,2000.0,72,72,0.85,0.1,0.2,0.044638,0.161863


In [ ]:
pred_rows = []
for h, ts in enumerate(hour_index[start_test_idx:start_test_idx + TEST_HOURS]):
    for j, cid in enumerate(cells['cell_id'].values):
        pred_rows.append({'dt': ts, 'cell_id': int(cid), 'x': float(cells.iloc[j]['x']), 'y': float(cells.iloc[j]['y']), 'pred_count': float(test_pred[h, j]), 'actual_count': float(test_true[h, j])})
pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(OUT_DIR / 'hourly_cell_predictions.csv', index=False)

cells_export = cells.copy()
cells_export['row'] = grid['row']
cells_export['col'] = grid['col']
cells_export.to_csv(OUT_DIR / 'grid_cells.csv', index=False)

metadata = {
    'target_crime': TARGET_CRIME or 'ALL',
    'study_start': str(study_start_dt),
    'grid_m': float(GRID_M),
    'patrol_tile_m': float(PATROL_TILE_M),
    'lookback_hours': int(LOOKBACK_HOURS),
    'test_hours': int(TEST_HOURS),
    'kde_sigma': float(KDE_SIGMA),
    'minx': float(grid['minx']),
    'miny': float(grid['miny']),
    'nrows': int(grid['nrows']),
    'ncols': int(grid['ncols']),
    'test_start': str(test_start),
    'end_hour': str(end_hour),
}
(OUT_DIR / 'grid_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

vmax = np.nanpercentile(np.concatenate([test_pred.ravel(), test_true.ravel()]), 99)
for h, ts in enumerate(hour_index[start_test_idx:start_test_idx + TEST_HOURS]):
    pred_img = np.full((grid['nrows'], grid['ncols']), np.nan, dtype=np.float32)
    actual_img = np.full((grid['nrows'], grid['ncols']), np.nan, dtype=np.float32)
    pred_img[grid['row'], grid['col']] = test_pred[h]
    actual_img[grid['row'], grid['col']] = test_true[h]
    tag = format_hour_name(ts)
    render_heatmap(pred_img, mask, f'Predicted Risk {tag}', HEATMAP_DIR / f'pred_{tag}.png', vmax=vmax)
    render_heatmap(actual_img, mask, f'Actual Counts {tag}', ACTUAL_HEATMAP_DIR / f'actual_{tag}.png', vmax=vmax)
print('Saved predicted heatmaps to', HEATMAP_DIR)
print('Saved actual heatmaps to', ACTUAL_HEATMAP_DIR)
print('Saved hourly predictions to', OUT_DIR / 'hourly_cell_predictions.csv')
print('Saved grid cells to', OUT_DIR / 'grid_cells.csv')
print('Saved grid metadata to', OUT_DIR / 'grid_metadata.json')


Saved predicted heatmaps to /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2/predicted_hourly_heatmaps
Saved actual heatmaps to /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2/actual_hourly_heatmaps
Saved hourly predictions to /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2/hourly_cell_predictions.csv
Saved grid cells to /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2/grid_cells.csv
Saved grid metadata to /content/drive/MyDrive/version_2/pointlevel_outputs_v16c_version_2/grid_metadata.json
